In [0]:
silver_stream = spark.readStream \
    .format("delta") \
    .table("clickstream_catalog.silver.clean_events")

In [0]:
from pyspark.sql.functions import window, count, sum as _sum, avg, col

gold_agg = silver_stream \
    .withWatermark("event_timestamp", "2 minutes") \
    .groupBy(
        window(col("event_timestamp"), "1 minute"),
        col("category"),
        col("event_type")
    ) \
    .agg(
        count("*").alias("event_count"),
        _sum("price").alias("total_revenue"),
        avg("price").alias("avg_price")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("category"),
        col("event_type"),
        col("event_count"),
        col("total_revenue"),
        col("avg_price")
    )

In [0]:
gold_checkpoint = "abfss://gold@itsmystorage.dfs.core.windows.net/clickstream_catalog/_checkpoints/gold_write/"

gold_query = gold_agg.writeStream \
    .format("delta") \
    .option("checkpointLocation", gold_checkpoint) \
    .outputMode("update") \
    .toTable("clickstream_catalog.gold.category_metrics_by_minute")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8330090261853904>, line 7
      1 gold_checkpoint = "abfss://gold@itsmystorage.dfs.core.windows.net/clickstream_catalog/_checkpoints/gold_write/"
      3 gold_query = gold_agg.writeStream \
      4     .format("delta") \
      5     .option("checkpointLocation", gold_checkpoint) \
      6     .outputMode("update") \
----> 7     .toTable("clickstream_catalog.gold.category_metrics_by_minute")

File /databricks/spark/python/pyspark/sql/connect/streaming/readwriter.py:738, in DataStreamWriter.toTable(self, tableName, format, outputMode, partitionBy, queryName, **options)
    729 def toTable(
    730     self,
    731     tableName: str,
   (...)
    736     **options: "OptionalPrimitiveType",
    737 ) -> "StreamingQuery":
--> 738     return self._start_internal(
    739         path=None,
    740         tableName=tableName,


In [0]:
from delta.tables import DeltaTable

def upsert_to_gold(microBatchDF, batchId):
    # Create the Gold table if it doesn't exist yet, from the first batch
    if not spark.catalog.tableExists("clickstream_catalog.gold.category_metrics_by_minute"):
        microBatchDF.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("clickstream_catalog.gold.category_metrics_by_minute")
        return

    gold_table = DeltaTable.forName(spark, "clickstream_catalog.gold.category_metrics_by_minute")

    gold_table.alias("target").merge(
        microBatchDF.alias("source"),
        """
        target.window_start = source.window_start AND
        target.category = source.category AND
        target.event_type = source.event_type
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

In [0]:
gold_checkpoint = "abfss://gold@itsmystorage.dfs.core.windows.net/clickstream_catalog/_checkpoints/gold_write/"

gold_query = gold_agg.writeStream \
    .option("checkpointLocation", gold_checkpoint) \
    .outputMode("update") \
    .foreachBatch(upsert_to_gold) \
    .start()

In [0]:
display(spark.sql("""
    SELECT * FROM clickstream_catalog.gold.category_metrics_by_minute 
    ORDER BY window_start DESC, event_count DESC
"""))

window_start,window_end,category,event_type,event_count,total_revenue,avg_price
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,home,page_view,4,null,null
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,fashion,page_view,4,null,null
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,books,checkout_start,3,656.14,218.71333333333334
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,beauty,page_view,3,null,null
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,toys,add_to_cart,3,626.22,208.74
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,home,purchase,2,693.8900000000001,346.94500000000005
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,home,checkout_start,2,557.0899999999999,278.54499999999996
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,electronics,page_view,2,null,null
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,books,add_to_cart,2,615.86,307.93
2026-07-08T07:58:00Z,2026-07-08T07:59:00Z,sports,purchase,1,262.02,262.02
